# Power Spectrum estimator example

This notebook demonstrates how to use the Power spectrum Statistics estimator from the ACM package.

This estimator is wrapped around `jaxpower` and will require the use of the `JaxpowerBackend` provided in `acm.estimators.galaxy_clustering.backends.jaxpower`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import load_estimator_parameters, make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.spectrum import PowerSpectrumMultipoles

setup_logging()

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)

# Instanciate class
estimator = PowerSpectrumMultipoles(
    backend='jaxpower',
    data_positions=data_positions,
    boxsize=boxsize,
    cellsize=5.0,
)
# Set the density contrast field with a smoothing radius
estimator.backend.set_density_contrast(
    smoothing_radius=10.0,  # Mpc/h
    resampler="tsc",
    interlacing=3,
    compensate=True,
)
# Compute the power spectrum multipoles with specified parameters
result = estimator.compute(
    edges={"step": 0.001},
    ells=(0, 2, 4),
    los=los,
    resampler="tsc",
    interlacing=3,
    compensate=True,
)

# Use the helper function to plot the result
fig, ax = estimator.plot(result)
ax.legend()
ax.grid(True, alpha=0.3)

*Note: not passing `resampler`, `interlacing` and `compensate` to either `set_density_contrast` or `compute` will default them to their default values. See https://github.com/adematti/jax-power/tree/main*

In [ ]:
# Access the data from the result object - see lsstypes docs

fig, ax = plt.subplots(figsize=(8, 6))

rebin=5
k = result.flatten(level=None)[0].coords("k")
for ell in (0, 2, 4):
    pole = result.get(ells=ell).select(k=slice(0, None, rebin))
    k = pole.coords("k")
    ax.plot(k, pole.value() * k, label=rf"$\ell={ell}$", alpha=0.8)

ax.set_xlabel(r"$k$ [h/Mpc]")
ax.set_ylabel(r"$k P(k)$ [(Mpc/h)$^3$]")
ax.set_title("Power Spectrum Multipoles")
ax.legend()
ax.grid(True, alpha=0.3)